## Clean version of extraction pipeline without the trial methods

### Method 1: Redacting the Table Area

approach # 1

Using pdfplumber - package, designed specifically for detecting tables.

In [ ]:
!pip install pymupdf pdfplumber pandas

In [ ]:
import fitz

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
doc = fitz.open("annual-report-ubs-group-2025.pdf")

In [ ]:
full_text = ""

for page in doc:
    full_text += page.get_text("text")

print(full_text[:1000])

Removing tables & noise

Natural Language Processing (NLP)

In [ ]:
!pip install spacy transformers torch
!python -m spacy download en_core_web_sm

In [ ]:
import re
import spacy
from transformers import pipeline

1. Filtering out tables

In [ ]:
num_pages = len(doc)
full_clean_text = ""

for page_num in range(num_pages):
    print(f"  -> Extracting Page {page_num + 1}/{num_pages}...")
    page_text = doc[page_num].get_text("text")

    cleaned_lines = []
# Use heuristic logic to skip lines that look like tables
    for line in page_text.split('\n'):
        numbers = re.findall(r"\d+", line)
        words = re.findall(r"[a-zA-Z]+", line)
        digit_ratio = sum(c.isdigit() for c in line) / max(len(line), 1)

 # If the line triggers any of these rules, it's a table row. Skip it.
        is_table = (
            len(numbers) >= 3 or
            digit_ratio > 0.3 or
            (len(numbers) >= 2 and len(words) >= 2) or
            len(line.split()) > 10 or
            "  " in line
        )

        if not is_table and line.strip():
            cleaned_lines.append(line.strip())

  # Stitch the surviving (non-table) lines together with spaces
    full_clean_text += " ".join(cleaned_lines) + " "

# Final cleanup to remove double spaces
full_clean_text = re.sub(r'\s+', ' ', full_clean_text)


**KEEP FOR FURTHER ANALYSIS**

2. LINGUISTIC ANALYSIS (spaCy)

In [ ]:
print("\n2. Running spaCy for Named Entity Recognition...")
nlp = spacy.load("en_core_web_sm")

# Process a chunk of the text to keep memory usage safe
doc_spacy = nlp(full_clean_text[:5000])

print("\n--- Key Financial Entities Found ---")
for ent in doc_spacy.ents:
    if ent.label_ in ["MONEY", "ORG", "DATE"]:
        print(f"{ent.text:25} | {ent.label_}")


# --- 3. TOPIC CLASSIFICATION (Zero-Shot) ---
print("\n3. Loading Classification Model...")
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

# Grab a random 200-character chunk from the middle of your clean text to test
sample_sentence = full_clean_text[2000:2200]
labels = ["Climate & ESG", "Financial Performance", "Corporate Governance", "Cybersecurity"]

result = classifier(sample_sentence, labels)
print("\n--- Topic Classification ---")
print(f"Text snippet: '{sample_sentence}...'")
print(f"Top Category: {result['labels'][0]} (Confidence: {result['scores'][0]*100:.1f}%)")

The model="facebook/bart-large-mnli" is a reference to a specific pre-trained AI model hosted on the Hugging Face Hub

**KEEP FOR FURTHER ANALYSIS**

4. FINANCIAL SENTIMENT ANALYSIS (FinBERT)

Sentence 1: Pulled from the "Key themes of 2021" section on Page 2.

Sentence 2: Pulled from the "Group Chief Executive’s review" on Page 10.

Sentence 3: Pulled from the "Our global reach" section on Page 6 (slightly paraphrased for length).

In [ ]:
print("\n4. Loading FinBERT Sentiment Model...")
fin_sentiment = pipeline("sentiment-analysis", model="ProsusAI/finbert")

# Analyzing three sentences manually extracted from your text stream
# (In a real pipeline, you would loop this over doc_spacy.sents)
test_sentences = [
    "Performance reflected an improvement in global economic conditions, which resulted in releases of expected credit loss allowances.",
    "Adjusted revenue was down 3%, due mainly to the impact of interest rate cuts.",
    "We maintained a strong capital, funding and liquidity position with a diversified business model."
]

print("\n--- Financial Sentiment Analysis ---")
for sent in test_sentences:
    res = fin_sentiment(sent)[0]
    print(f"[{res['label']:^10}] | {sent}")

print("\nPipeline Complete")

In [ ]:
print(full_clean_text[:3000])

In [ ]:
with open("clean_ubs_text_2025.txt", "w", encoding="utf-8") as f:
    f.write(full_clean_text)

print("\n--- FILE SAVED ---")